# Simulating a Disease-Conditioned Human Cortex
## A Working Tutorial on a Multi-Area Spiking Network Model

The model used here is a **spiking network model of one human cerebral hemisphere**, built to ask a specific
question: *what does amyloid-beta pathology do to the dynamics of the cortex as a whole?* It extends
the **Human Multi-Area Model (HuMAM)** from Forschungszentrum Julich with a disease neuron
population, oscillatory stimulation, and an analysis pipeline that ends in signals you can put next
to EEG and fMRI.

At full scale the model has **3.47 million neurons** and **42.8 billion synapses**. This tutorial
runs the same model, downscaled by a factor of 2000, on a laptop - and then shows you how to send
the full-scale version to a cluster.

---

### What you will do

1. Understand what the model represents and what its parameters mean
2. Read and edit an experiment file - the model's entire control panel
3. Understand **downscaling**, and why a careless downscale produces a *silent* network
4. Run the full three-stage pipeline on a healthy cortex
5. Read the ~200 figures it produces, and learn which ones to check first
6. Repeat with amyloid pathology and compare
7. Decide, with a seed check, which differences are real
8. Send the full-scale model to a cluster

**If you meet any problem, feel free to raise your hand!**


---
## Part 0 - Quick start

### 0.1 Get your environment ready.

We have a environment package list prepared in this repository.
Create a new virtual environment quickly with one-line command.

Run one of these commands below.
```shell
mamba create -f environment.yaml 
```
or
```shell
conda create -f environment.yaml 
```
Remember to activate your environment.

### 0.2 Does NEST work on my computer?
If you install the pre-compiled NEST package properly, take a simple test.
Run ```python -c "import nest"``` and you are supposed to see something like this:
```shell
              -- N E S T --
  Copyright (C) 2004 The NEST Initiative

 Version: 3.8.0
 Built: May  5 2025 12:54:52

 This program is provided AS IS and comes with
 NO WARRANTY. See the file LICENSE for details.

 Problems or suggestions?
   Visit https://www.nest-simulator.org

 Type 'nest.help()' to find out more about NEST.

bundled model   : /YOUR/PATH/TO/Cortical-Model-Tutorial/model
Results go to   : /YOUR/PATH/TO/Cortical-Model-Tutorial/results
NEST            : 3.8.0
numpy / pandas  : 2.2.6 / 2.2.3
CPU cores       : 16
```

Congradulations! A MPI-free NEST is installed and works well (in most time) !

### 0.3 It looks like ... crashed?

A spiking simulation's memory is dominated by **synapses**, not neurons. Measured on this model
(Apple Silicon, NEST 3.8, 4 threads):

```
peak RAM  ~=  0.4 GB  +  128 bytes x (number of synapses)
```

**Laptop notes: Plug in - both macOS and Windows down-clock hard on battery.**
Expect fans: a 5 s simulation is ~19 s of full-throttle multicore compute. **If a cell seems to hang for minutes, you are
probably swapping; stop it and consider a smaller network (Part 2)**.

### 0.4 Final check
Before click the button ```Run All```, check if your notebook uses the proper ipykernel (nest3.8)!
```

In [ ]:
# One-click check
# This is supposed to exit without error.

import os
import subprocess
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

TUTORIAL = Path.cwd().resolve()
EXPERIMENTS = TUTORIAL / "experiments"
RESULTS = TUTORIAL / "results"
(RESULTS / "log").mkdir(parents=True, exist_ok=True)


# The model itself ships inside this tutorial, in ./model - there is nothing to
# clone, no environment variable to set and no version to match.
MODEL_ROOT = TUTORIAL / "model"
if not (MODEL_ROOT / "src" / "snakemake_network.py").exists():
    raise SystemExit(
        f"The bundled model is missing from {MODEL_ROOT}.\n"
        "Re-clone this tutorial repository, or run the notebook from its own folder.")

# The tutorial's own helpers (the drive-calibration tool doubles as a spike reader).
sys.path.insert(0, str(TUTORIAL / "tools"))

import nest                                     # fails early if NEST is missing

print(f"bundled model   : {MODEL_ROOT}")
print(f"Results go to   : {RESULTS}")
print(f"NEST            : {nest.__version__}")
print(f"numpy / pandas  : {np.__version__} / {pd.__version__}")
print(f"CPU cores       : {os.cpu_count()}")

bundled model   : /Users/nozomizore1974/Codes/Cortical-Model-Tutorial/model
Results go to   : /Users/nozomizore1974/Codes/Cortical-Model-Tutorial/results
NEST            : 3.8.0
numpy / pandas  : 2.2.6 / 2.2.3
CPU cores       : 16


---

# Part 1 - Unveil the model

## 1.1 Architecture

```shell
   MACRO   34 cortical areas, Desikan-Killiany atlas, one hemisphere
             |  inter-area connection strength from HCP diffusion tractography
             v
   MESO    4 layers per area (II/III, IV, V, VI)
             |  layer-resolved neuron densities, von Economo & Koskinas
             |  laminar target patterns from predictive connectomics
             v
   MICRO   2 populations per layer: excitatory (E) and inhibitory (I)
             |  leaky integrate-and-fire neurons, exponential synaptic currents
             v
           34 x 4 x 2 = 272 populations, 254 of them non-empty
           3.47 M neurons, 42.8 G internal synapses at full scale
```

Each area is modelled as a **1 mm^2 patch** of cortex with that area's own thickness and cell
density - the classic cortical-column idealisation, repeated 34 times and wired together.

## 1.2 Parameters

| Ingredient | Source | In the repository |
|---|---|---|
| Layer-resolved neuron **densities** | von Economo & Koskinas cytoarchitectonic atlas, mapped to Desikan-Killiany | `experimental_data/voneconomokoskinas/` |
| **E/I ratio** per layer | Lichtman / Binzegger electron-microscopy counts | `experimental_data/fraction_EI/` |
| **Inter-area connectivity and distances** | HCP diffusion MRI tractography | `experimental_data/hcp_dti/` |
| **Laminar target patterns** | predictive connectomics (Schmidt et al. 2018) | `model/src/data_preprocessing/connectivity.py` |
| **Synapse density** | 6.6e8 per mm^3 (Cano-Astorga et al. 2021) | `model/src/default_net_params.py` |
| **Reference resting-state fMRI** | Senden 7T, 19 participants (No raw data) | `experimental_data/senden/` |
| **Neuron parameters** | Allen Brain Cell Atlas GLIF fits | the experiment files |
| **Amyloid spatial distribution** | amyloid-PET (ADNI participant), DK-parcellated | `experimental_data/abeta/` |

Nothing here is fitted to reproduce a target output. That is the whole point: the dynamics are a
*prediction*, and the comparison with EEG or fMRI is the test.

> **Compass - for machine learning people.** There is no training set, no loss, no backprop. The
> architecture is measured; the weights follow from a single postsynaptic-potential amplitude plus
> scaling rules. What you tune is a handful of global gains, and you judge the result by whether
> emergent statistics match independent measurements. This is the generative-model tradition, not
> the function-approximation one.

## 1.4 What a neuron is here

Every neuron is a **leaky integrate-and-fire** unit with exponentially decaying synaptic currents
(`iaf_psc_exp` in NEST). Its membrane potential follows

$$\tau_m \frac{dV}{dt} = -(V - E_L) + \frac{\tau_m}{C_m}\, I_{\mathrm{syn}}(t) + \frac{\tau_m}{C_m}\, I_e$$

and when $V$ reaches $V_{\mathrm{th}}$ the neuron emits a spike, is reset to $V_{\mathrm{reset}}$
and held there for a refractory time $t_{\mathrm{ref}}$. Each incoming spike adds a current pulse
that decays with $\tau_{\mathrm{syn}}$.

| Parameter | Excitatory | Inhibitory | Meaning |
|---|---|---|---|
| $E_L$ | -70 mV | -70 mV | resting potential |
| $V_{\mathrm{th}}$ | -45 mV | -45 mV | firing threshold |
| $V_{\mathrm{reset}}$ | -70 mV | -70 mV | post-spike potential |
| $C_m$ | 220 pF | 100 pF | membrane capacitance |
| $\tau_m$ | 10 ms | 10 ms | membrane time constant |
| $\tau_{\mathrm{syn}}$ | 2 ms | 2 ms | synaptic current decay |
| $t_{\mathrm{ref}}$ | 2 ms | 2 ms | refractory period |

The number that matters most below is $V_{\mathrm{th}} - E_L = 25$ mV: **the distance a neuron must
be driven to fire**. Amyloid shrinks it to 15 mV (Part 4), and downscaling changes how far the drive
actually reaches (Part 3).

$I_e$ is a constant current. It is zero in the full-scale model and carries the downscaling
compensation otherwise - remember it, it reappears in Part 3.

## 1.5 What a connection is

Three things define the recurrent wiring:

- **In-degree** $K_{ij}$ - how many synapses a neuron in population $i$ receives from population
  $j$. At full scale a neuron has about **12 500** recurrent inputs. This is the dominant number in
  the model; everything else is secondary to it.
- **Weight** $J_{ij}$ - specified as a postsynaptic *potential* (0.1 mV for excitatory connections,
  5x larger and negative for inhibitory ones) and converted internally to a current amplitude.
- **Delay** - 1.5 ms locally, and distance divided by a 3.5 mm/ms axonal speed between areas, each
  drawn from a clipped lognormal distribution.

Inter-area connectivity uses **predictive connectomics**: tractography gives the strength of an
area-to-area projection, and the *relative cytoarchitectonic difference* between the two areas
predicts which layers it targets. Areas of similar cell density connect more densely, and the
direction of the density gradient decides whether a projection behaves as feedforward (terminating
in layer IV) or feedback (terminating outside it). `FLN = 0.86` of all synapses stay within an area.

Each population also receives **Poisson background input** standing in for everything outside the
modelled hemisphere - the other hemisphere, thalamus, neuromodulation. Its strength is the parameter
`eta_ext`, expressed as a multiple of the distance to threshold. It is the single most important
dial in the model, and Part 3 is about setting it correctly.

## 1.6 What comes out, and what it maps onto

| Simulated signal | How it is built | Experimental counterpart |
|---|---|---|
| **Spike trains** | recorded directly | extracellular unit recordings |
| **Firing rate, CV ISI, Lv** | per population | spike-train statistics from electrode arrays |
| **LFP proxy** | synaptic currents, or spikes convolved with the synaptic kernel | intracranial LFP; the generator of M/EEG |
| **Power spectra, coherence, PLV** | from the LFP proxy | quantitative EEG, MEG connectivity |
| **BOLD** | area firing rate through the Balloon-Windkessel model | fMRI time course |
| **Functional connectivity** | pairwise BOLD correlation | resting-state FC matrix |

> **Compass - for neuroimaging readers.** The chain from spikes to your data is explicit here, and
> every link is inspectable. The LFP proxy is *not* forward-modelled onto a scalp, so absolute band
> ratios are not comparable to clinical qEEG; differences between conditions on the same proxy are.

---

# Part 2 - The Control Panel: an Experiment File

An experiment is **one Python file with three dictionaries**. There is no other interface.
Anything you leave out keeps the repository default from `model/src/default_net_params.py`,
`model/src/default_sim_params.py` and `model/src/default_ana_params.py`.

| Dictionary | Answers | Fixes |
|---|---|---|
| `net_params` | *What is the network?* | size, scaling, drive, connection gains, pathology |
| `sim_params` | *What happens to it?* | simulated duration, seed, recorders, stimulation |
| `ana_params` | *What is measured afterwards?* | analysis windows, which proxies, figure settings |

The split matters because it matches the three pipeline stages, and each stage's output directory is
named by a hash of its own dictionary. Change `ana_params` and you re-run only the analysis; change
`net_params` and everything downstream is rebuilt.

Let us read the file this tutorial ships.

In [ ]:
print((EXPERIMENTS / "tutorial_CN.py").read_text())

### The knobs that actually matter

Most defaults you will never touch. These you will:

| Parameter | What it does | Typical values |
|---|---|---|
| `N_scaling` | fraction of neurons kept | 1.0 full, 0.005 laptop |
| `K_scaling` | fraction of in-degree kept | 1.0 full, 0.1 downscaled |
| `input_params["eta_ext"]` | external drive, in units of the distance to threshold | 1.1 full scale, ~3.45 downscaled |
| `scaling_factors_recurrent["cc_scalingEtoE"]` | gain on long-range excitation | 1.0-3.0; higher = more inter-area coupling, closer to instability |
| `scaling_factors_recurrent["cc_scalingEtoI"]` | gain on long-range excitation onto interneurons | usually 2x the E->E value, and what keeps the network stable |
| `Abeta`, `Abeta_ratio_E` | pathology on/off and its spatial distribution | `False`, or a scalar / dict / PET-derived CSV |
| `t_sim` | simulated biological time | 5000 ms here; 10-100 s on a cluster |
| `master_seed` | the random draw | any integer - **change it to measure your own noise floor** |
| `rec_dev` | recorders | `["spike_recorder"]`; add `"I_syn"` for an exact LFP at a large disk cost |
| `stimuli` | oscillatory current injection | `ac_generator` per `(area, layer, population)` |

### Choosing a size for your machine

`N_scaling` and `K_scaling` are **not** interchangeable:

- lowering **`K_scaling`** changes the dynamics - each neuron sees fewer inputs - and must be
  compensated (Part 3);
- lowering **`N_scaling`** mostly adds finite-size noise and leaves the mean-field behaviour intact.

So the right way to fit the model onto a laptop is to hold `K_scaling` at the value the project
validates (0.1) and push `N_scaling` down. Edit `experiments/tutorial_CN.py` if the default does not
fit your machine:

| `N_scaling` | Neurons | Synapses | Peak RAM (predicted / measured) | Simulate 5 s |
|---|---|---|---|---|
| 0.002 | 6.9 k | 8.6 M | 1.4 GB / **1.3 GB** | ~7 s |
| **0.005** (default) | **17 k** | **21 M** | **2.9 GB / 3.1 GB** | **~19 s** |
| 0.02 | 69 k | 86 M | 10.6 GB / - | ~75 s |
| 1.0 (with `K_scaling` 1.0) | 3.47 M | 42.8 G | ~5 TB / - | cluster only |

> **Compass - for CS readers.** The three dictionaries are the model's public API, and the hash of
> each is the cache key of a stage. It is content-addressed storage applied to scientific
> parameters: identical inputs give an identical path, so you cannot silently overwrite a run with
> different parameters, and re-running costs nothing.

---

# Part 3 - Downscaling, and the Trap in It

This is the part to read carefully. Skipping it is the quickest way to produce beautiful figures of
a network that is not doing what you think.

## 3.1 What downscaling does to the input

A neuron's input is characterised by two numbers: the **mean** and the **variance** of its total
synaptic drive. For a population receiving in-degree $K$ with weight $J$ from presynaptic neurons
firing at rate $\nu$:

$$\mu \;\propto\; \sum_j J_j K_j \nu_j
\qquad\qquad
\sigma^2 \;\propto\; \sum_j J_j^2 K_j \nu_j$$

Keep only a fraction $k$ of the in-degree and the two collapse together - unless you also raise the
weights. The standard remedy (van Albada et al. 2015) is

$$J' = \frac{J}{\sqrt{k}}, \qquad K' = k\,K$$

which gives

$$\sigma'^2 \;\propto\; \frac{J^2}{k}(kK)\nu = \sigma^2 \qquad\text{(variance preserved exactly)}$$

$$\mu' \;\propto\; \frac{J}{\sqrt{k}}(kK)\nu = \sqrt{k}\,\mu \qquad\text{(mean shrinks by } \sqrt{k})$$

Fluctuations survive. **The mean drive does not.** So a downscaled network also needs a compensating
DC current carrying the missing $(1-\sqrt{k})\mu$ - that is the $I_e$ from Part 1.4, and it is why a
downscaled run requires `fullscale_rates`: the compensation has to know what rates it is
compensating for.

> **Compass - for ML readers.** The closest thing you know is variance-preserving initialisation
> (Xavier/He): when fan-in changes, rescale weights by $1/\sqrt{\text{fan-in}}$ to hold activation
> variance. The difference is that a spiking network also has a *mean* operating point relative to a
> threshold, and that mean is not preserved by the same rescaling. Get it wrong and the network does
> not merely perform badly - it goes silent.

## 3.2 The trap

The model applies exactly this compensation in `Network.extraDCforScaledNetwork()`. But it evaluates
the compensation term from the *already-scaled* weights and in-degrees, so it restores
$(1-\sqrt{k})\sqrt{k}\,\mu$ instead of $(1-\sqrt{k})\,\mu$ - about **28%** of the missing drive at
$k = 0.1$ rather than 90%.

The consequence is not subtle. At `K_scaling = 0.1` and the repository default `eta_ext = 1.1`, the
mean drive reaches only **44%** of the distance to threshold, with fluctuations of about 1.5 mV
against a 14 mV gap. The network emits nothing beyond the handful of spikes caused by its random
initial membrane potentials, and then stays silent for the rest of the run.

This is worth stating plainly because it is easy to miss: the repository's own `local_*` experiments
*do* produce activity, but only because they enable amyloid neurons, whose threshold is 10 mV lower.
Set `Abeta_ratio_E = 0` in those files and the same configuration falls silent.

## 3.3 The fix: recalibrate the drive

The clean remedy is to re-tune `eta_ext` for the downscaled network - exactly as you would re-tune a
learning rate after changing batch size. The target is the working range the project uses elsewhere
**excitatory 3-5 Hz, inhibitory 5-15 Hz**.

`tools/eta_ext_search.py` measures it for you:

```bash
python tools/eta_ext_search.py                    # simulate + bisect
python tools/eta_ext_search.py --method analytic  # instant closed-form estimate
python tools/eta_ext_search.py --method theory    # NNMT mean-field, ~5 s per point
python tools/eta_ext_search.py --help
```

For the network this tutorial uses (`N_scaling=0.005, K_scaling=0.1`) it reports:

| | `eta_ext` | mean drive vs threshold | result |
|---|---|---|---|
| repository default | 1.1 | 44% | **silent** |
| restores the full-scale mean | 2.54 | 110% | ~2 Hz, below target |
| **used in this tutorial** | **3.45** | **152%** | **E 3.0 Hz, I 12.1 Hz - inside the target band** |
| network first wakes up | 1.38 | 57% | barely alive |

That is why `tutorial_CN.py` sets `eta_ext = 3.45` rather than the repository default.

## 3.4 The rule to take away

> **A downscaled network is a different model.** Verify its working point - the firing rates -
> *before* you read any other figure. If the rates are zero, or 50 Hz, nothing downstream means
> anything.

It is an excellent instrument for building intuition, developing analysis, and exploring parameter
directions. It is not a substitute for the full-scale run when you want a number. Part 8 covers that.

---

# Part 4 - Amyloid-Beta in the Model

## 4.1 The biology, briefly

Amyloid-beta aggregates into extracellular plaques and is one of the two defining pathologies of
Alzheimer's disease. For a *dynamical* model the important point is that it does not merely kill
neurons: well before cell death, surviving neurons in amyloid-burdened tissue show altered membrane
properties and a **lowered effective firing threshold**, and both human and mouse work reports
**network hyperexcitability** - including a several-fold increase in seizure incidence in patients
(Palop & Mucke 2016).

## 4.2 The parameter mapping

The implementation does not represent amyloid chemistry. It replaces a configurable **fraction of the excitatory
neurons in each population** with units whose parameters are fitted to Allen Brain Cell Atlas GLIF
recordings of amyloid-burdened cells:

| | healthy E neuron | amyloid-affected E neuron |
|---|---|---|
| $E_L$ | -70 mV | **-65 mV** |
| $V_{\mathrm{th}}$ | -45 mV | **-50 mV** |
| **distance to threshold** | **25 mV** | **15 mV** |
| $C_m$, $\tau_m$, $t_{\mathrm{ref}}$ | 220 pF, 10 ms, 2 ms | unchanged |

A 40% shorter path to threshold, and nothing else. Connectivity, weights, delays and drive are
identical between `tutorial_CN.py` and `tutorial_AD.py`, so every difference in the results traces
back to this substitution.

## 4.3 From an amyloid-PET scan to a parameter

`Abeta_ratio_E` accepts four forms, in increasing specificity:

| Form | Example | Meaning |
|---|---|---|
| scalar | `0.15` | 15% of excitatory neurons everywhere |
| per area | `{"precuneus": 0.3, "default": 0.05}` | per-area fractions with a fallback |
| per area, per layer | `{"precuneus": {"II/III": 0.3}}` | laminar resolution |
| CSV path | `experimental_data/abeta/002_S_0729_abeta_ratio.csv` | a participant's amyloid-PET map |

The CSV form is the one that makes this a patient-specific model: an individual's **amyloid-PET SUVR
map**, parcellated into Desikan-Killiany areas, becomes a per-area disease burden. The repository
bundles one derived from ADNI participant `002_S_0729`, and
`model/experimental_data/abeta/generate_abeta_csv.py` shows how such a file is produced.

In [ ]:
abeta = pd.read_csv(MODEL_ROOT / "experimental_data" / "abeta" / "002_S_0729_abeta_ratio.csv")
per_area = (abeta[abeta["area"] != "default"]
            .set_index("area")["ratio"].sort_values(ascending=False))

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.bar(range(len(per_area)), per_area.values,
       color=plt.cm.YlOrRd(0.25 + 0.75 * per_area.values / per_area.max()))
ax.set_xticks(range(len(per_area)))
ax.set_xticklabels(per_area.index, rotation=90, fontsize=8)
ax.axhline(per_area.mean(), ls="--", c="k", lw=1, label=f"mean {per_area.mean():.3f}")
ax.set_ylabel("fraction of E neurons affected")
ax.set_title("Amyloid burden per area - ADNI participant 002_S_0729 (amyloid-PET, DK parcellation)")
ax.legend()
plt.tight_layout(); plt.show()

print(f"range           : {per_area.min():.3f} - {per_area.max():.3f}")
print(f"heaviest areas  : {', '.join(per_area.head(5).index)}")
print(f"lightest areas  : {', '.join(per_area.tail(5).index)}")

**Read the spatial pattern.** The burden concentrates in posterior default-mode hubs and
medial/orbitofrontal cortex; primary sensory and motor areas are comparatively spared. That is the
canonical early-amyloid (Thal phase) distribution - and it is exactly why a network model is worth
building. The *same* amount of pathology produces different dynamics depending on **where in the
connectome** it lands.

---

# Part 5 - Running the Model

## 5.1 The three stages

Every run is the same three commands, each consuming the experiment file and producing a
hash-named directory:

| Stage | Does | Outputs | Cost here |
|---|---|---|---|
| **1. network** | reads the anatomical data, builds neuron numbers, the connectome, weights and delays, applies downscaling and its DC compensation | `results/<net_hash>/` | ~65 s |
| **2. simulate** | instantiates the network in NEST, connects it, injects Poisson background and any stimuli, integrates at 0.1 ms | `results/<net_hash>/<sim_hash>/spikes/` | ~19 s |
| **3. analyse** | runs `fullAnalysis()`: spike statistics, LFP proxy, coherence, PLV, BOLD, functional connectivity, and ~200 figures | `results/<net_hash>/<sim_hash>/<ana_hash>/` | ~110 s |

In [ ]:
def run_stage(stage, experiment, threads=4, quiet=False):
    """Run one pipeline stage for `experiment`, with everything outside the clone."""
    exp = EXPERIMENTS / f"{experiment}.py"
    base = EXPERIMENTS / experiment
    log = RESULTS / "log" / f"{experiment}_{stage}.log"
    cmd = {
        "network":  ["python", "src/snakemake_network.py", str(exp),
                     f"{base}.network_hash", str(log)],
        "simulate": ["python", "src/snakemake_simulation.py", str(exp),
                     f"{base}.network_hash", f"{base}.simulation_hash",
                     str(threads), str(log)],
        "analyse":  ["python", "src/snakemake_analysis.py", str(exp),
                     f"{base}.network_hash", f"{base}.simulation_hash",
                     f"{base}.analysis_hash", str(MODEL_ROOT), str(log)],
    }[stage]

    t0 = time.time()
    # cwd = the model repository: it resolves its data paths from the cwd.
    proc = subprocess.run(cmd, cwd=MODEL_ROOT, capture_output=True, text=True)
    elapsed = time.time() - t0
    if proc.returncode != 0:
        print(proc.stdout[-2000:])
        print(proc.stderr[-4000:])
        raise RuntimeError(f"stage '{stage}' failed for {experiment}")
    if not quiet:
        print(f"  {stage:9s} {elapsed:6.1f} s")
    return elapsed


def hashes(experiment):
    """The hashes written so far, as a dict."""
    base = EXPERIMENTS / experiment
    out = {}
    for stage in ("network", "simulation", "analysis"):
        f = Path(f"{base}.{stage}_hash")
        if f.exists():
            out[stage] = f.read_text().strip()
    return out


def run_dir(experiment):
    """The analysis directory of a completed run."""
    h = hashes(experiment)
    return RESULTS / h["network"] / h["simulation"] / h["analysis"]


def sim_dir(experiment):
    h = hashes(experiment)
    return RESULTS / h["network"] / h["simulation"]


print("Pipeline helpers ready.")

In [ ]:
# ~3 minutes. Progress appears as each stage finishes; detailed logs are in results/log/.
print("tutorial_CN  (healthy cortex)")
total = 0.0
for stage in ("network", "simulate", "analyse"):
    total += run_stage(stage, "tutorial_CN")
print(f"  {'total':9s} {total:6.1f} s\n")

for stage, h in hashes("tutorial_CN").items():
    print(f"{stage:11s} hash  {h}")

In [ ]:
# What the pipeline produced.
d = run_dir("tutorial_CN")
print(f"analysis directory: {d.relative_to(TUTORIAL)}\n")

plots = sorted(p for p in (d / "plots").glob("*.png"))
groups = sorted(p for p in (d / "plots").iterdir() if p.is_dir())
data = sorted(p for p in d.glob("*.pkl"))

print(f"{len(plots)} summary figures, e.g.")
for p in plots[:6]:
    print("   ", p.name)
print(f"\n{len(groups)} per-area figure sets:")
for g in groups:
    print(f"    {g.name}/  ({len(list(g.iterdir()))} areas)")
print(f"\n{len(data)} result files you can load directly:")
for p in data:
    print("   ", p.name)

size = sum(f.stat().st_size for f in d.rglob("*") if f.is_file()) / 1024**2
print(f"\ntotal {size:.0f} MB")

---

# Part 6 - Reading the Output

The analysis stage produced roughly 200 figures. That is a lot, and most of them you will not look
at on any given day. This part walks the ones that matter, in the order you should consult them.

**The order is not arbitrary.** Figures 1 and 2 tell you whether the run is worth interpreting at
all. Only if they look right do the rest mean anything.

In [ ]:
def show(experiment, name, width=900):
    """Display one figure produced by the analysis stage."""
    path = run_dir(experiment) / "plots" / name
    if not path.exists():
        print(f"not found: {name}")
        return
    display(Image(filename=str(path), width=width))


def rates(experiment):
    """Population firing rates (Hz), MultiIndex (area, layer, pop)."""
    return pd.read_pickle(sim_dir(experiment) / "rates.pkl")


r = rates("tutorial_CN")
pop = r.index.names[-1]
print(f"mean excitatory rate : {r.xs('E', level=pop).mean():6.2f} Hz   (target 3-5)")
print(f"mean inhibitory rate : {r.xs('I', level=pop).mean():6.2f} Hz   (target 5-15)")
print(f"silent populations   : {(r == 0).sum()} of {len(r)}")
print(f"fastest population   : {r.idxmax()}  at {r.max():.1f} Hz")

## 6.1 Check first: is the network in a sensible state?

Three questions, in this order:

1. **Are the rates plausible?** Excitatory populations at a few Hz, inhibitory ones higher
   (interneurons fire faster). Zero means the working point is wrong - go back to Part 3. Tens of Hz
   everywhere means the network has run away.
2. **Is the firing irregular?** The awake resting cortex is *asynchronous irregular*: neurons fire
   at random-looking intervals, and the population does not beat in unison. The coefficient of
   variation of inter-spike intervals should sit near 0.7-1.0. Much below 0.5 means clock-like
   firing; a raster full of vertical stripes means synchrony.
3. **Are neurons decorrelated?** Pairwise spike correlations should be close to zero. Large positive
   correlations mean the whole population is being driven together, which in this model usually
   signals instability rather than a discovery.

The pipeline's summary figure answers all three at once.

In [ ]:
show("tutorial_CN", "figure_spike_statistics.png", width=980)

**How to read it.** Panels **A-C** are raster plots of three areas over the last second: each dot is
a spike, rows are neurons grouped by layer and type (blue excitatory, red inhibitory). You want a
sprinkle of pepper, not vertical stripes. Panel **D** is the distribution of firing rates per
population, **E** the CV of inter-spike intervals, **F** the pairwise correlation coefficients.

Note how different the three areas look. That is not noise: areas differ in thickness, cell density
and how strongly they are embedded in the connectome, so they sit at different operating points.
Pericalcarine cortex (primary visual) is dense and busy; caudal anterior cingulate is sparse and
quiet.

If D shows a spike at zero, E collapses toward 0, or F spreads far from zero, stop and fix the
working point before reading anything else.

In [ ]:
show("tutorial_CN", "boxplot.png", width=620)
show("tutorial_CN", "cv_isi.png", width=620)

These give the same information split by layer and cell type, pooled over all 34 areas. Two things
are worth reading off them.

**Inhibitory populations fire several times faster than the excitatory ones in the same layer.** That
is a basic cortical regularity and a useful sanity check - if it fails, inhibition is not doing its
job and the network is probably on its way to instability.

**The layers are not equal.** In this configuration layer IV is by far the busiest (excitatory
populations around 6-7 Hz), while layers II/III and VI are the quietest (well under 2 Hz). Compare
that ordering with the one reported for the cortical microcircuit it inherits from (Potjans &
Diesmann 2014, where layer V excitatory leads): the ordering depends on the working point, and
checking it is one way to tell whether your `eta_ext` and connection gains put the model where you
intended.

## 6.2 Where is the activity, anatomically?

In [ ]:
show("tutorial_CN", "spiking_overview_same_y_axis_True.png", width=980)

One panel per area, all on the same vertical scale, showing the population rate over time. This is
the figure to use when you want to know *which areas* a manipulation affected - and in Part 7 we
will do exactly that.

A flat trace means a steady asynchronous state. Slow drifts shared across many areas would indicate
a global oscillation, which at this network size is more often a sign of trouble than of biology.

## 6.3 Rhythms: the LFP proxy and its spectrum

Spikes are not what an EEG measures. That signal is generated by **transmembrane currents**,
dominated by synaptic currents in pyramidal dendrites. The LFP proxy sums the synaptic currents
entering a population, with excitatory input as a sink and inhibitory input as a source, and
z-scores the result per `(area, layer)` - so only *shape* is meaningful, never absolute amplitude.

In [ ]:
show("tutorial_CN", "lfp_proxy_psd_layer_II_III.png", width=980)

Power spectra of the layer II/III LFP proxy, one panel per area. The broad peak in the gamma range
is generated by the **pyramidal-interneuron gamma (PING) loop** - the excitatory-inhibitory feedback
motif present in every layer. Its frequency is set by how fast excitation recruits inhibition, so it
moves when you change rates, weights or thresholds.

Two honest caveats before you compare this with an EEG:

- **This model has no thalamus**, and cortical alpha depends heavily on thalamocortical loops. Read
  the alpha band as "the low-frequency side of the cortical spectrum", not as the occipital rhythm.
- **There is no forward model**: no volume conduction, no scalp. Absolute band fractions are not
  comparable to clinical qEEG numbers. Differences between conditions on the same proxy are.

In [ ]:
show("tutorial_CN", "spectrogram/lfp_spectrogram_precuneus.png", width=820)

A time-frequency view of the same signal for one area, one panel per layer. Use it to check
**stationarity**: the spectral content should look statistically the same throughout. Bands that
appear and disappear mean the network is drifting between states, and a spectrum averaged over the
whole window would then be meaningless.

The per-area versions of every LFP figure live in `plots/lfp/`, `plots/psd/`, `plots/spectrogram/`
and `plots/wavelet/` - 34 areas each.

## 6.4 Long-range synchrony

In [ ]:
show("tutorial_CN", "lfp_area_coherence.png", width=780)

Magnitude-squared coherence between the area-level LFP proxies of every pair of areas: how
consistently two areas' rhythms keep a fixed phase relationship, frequency by frequency. It is the
simulated counterpart of MEG/EEG connectivity analysis.

The companion figures are `lfp_coherence_fulllayer.png` (the full `(area, layer)` x `(area, layer)`
matrix), the per-layer versions, and `lfp_plv_fulllayer.png`, which measures **phase locking**
independently of amplitude - useful precisely because coherence can be inflated by a shared
amplitude envelope.

## 6.5 Haemodynamics: BOLD and functional connectivity

Everything so far lives on a millisecond timescale. fMRI is ~1000x slower. The bridge is the
**Balloon-Windkessel model** (Buxton 1998; Friston 2000): a small ODE system turning neural activity
into vasodilation, blood flow, blood volume, deoxyhaemoglobin, and finally a BOLD signal. Correlating
area BOLD time courses pairwise gives a **simulated functional connectivity matrix**, directly
comparable to a resting-state FC matrix.

In [ ]:
show("tutorial_CN", "bold_overview.png", width=980)
show("tutorial_CN", "simulated_fc.png", width=560)
show("tutorial_CN", "experimental_fc.png", width=560)

The last two are the model's FC and the **Senden 7T resting-state dataset** (19 participants, DK
parcellation) shipped with the repository. Comparing them is the standard check for a whole-brain
model, and the number reported in a paper.

**Expect a weak correlation from this run, and do not be alarmed.** Published whole-brain models
report r in the 0.3-0.6 range - with the full-scale model, minutes of simulated time, and parameters
tuned for that objective. Here we have **5 seconds** of biological time, which is a handful of BOLD
samples; the simulated matrix is dominated by transients. Treat it as a demonstration of the
*pipeline*, not as a validation of the model.

To get a meaningful FC number you need, in order of importance: **much longer runs** (100 s+), **full
or near-full scale**, and **several seeds**. All three are cluster jobs - Part 8.

## 6.6 The numbers behind the figures

Every figure has a data file next to it, so you never have to re-derive anything:

| File | Contains |
|---|---|
| `rates.pkl` | firing rate per population *(next to the spikes, one level up)* |
| `lfp_proxy_kernel.pkl` | LFP proxy, `(area, layer)` x time |
| `lfp_coherence.pkl`, `lfp_plv.pkl` | coherence and phase-locking matrices |
| `lfp_proxy_area.pkl`, `lfp_area_correlation.pkl` | area-level LFP and its correlation matrix |
| `bold_signal.pkl` | BOLD time course per area |
| `rate_histogram.pkl`, `rate_histogram_areas.pkl` | binned population rates |
| `input_current.pkl` | area-averaged synaptic input current |

All are pandas objects; `pd.read_pickle` is all you need.

In [ ]:
lfp = pd.read_pickle(run_dir("tutorial_CN") / "lfp_proxy_kernel.pkl")
print(f"LFP proxy : {lfp.shape[0]} (area, layer) signals x {lfp.shape[1]} samples "
      f"at {1000 / (lfp.columns[1] - lfp.columns[0]):.0f} Hz")

busiest = (rates("tutorial_CN").xs("E", level=pop)
           .groupby(level="area").mean().sort_values(ascending=False))
print("\nmean excitatory rate, five busiest and five quietest areas (Hz):")
print(pd.concat([busiest.head(5), busiest.tail(5)]).round(2).to_string())

## 6.7 A checklist to reuse

Every time you look at a new run, in this order:

1. `rates.pkl` - are E and I rates in a plausible band, and is anything silent?
2. `figure_spike_statistics.png` - irregular, decorrelated, no stripes?
3. `spiking_overview_same_y_axis_True.png` - stationary, no runaway area?
4. `spectrogram/` - stationary in time-frequency too?
5. *Only now* the science figures: spectra, coherence, BOLD, FC.
6. Before claiming any effect: **re-run with another seed** (Part 7.3).

---

# Part 7 - The Experiment: Healthy vs Amyloid

Now the actual scientific use of the model. `experiments/tutorial_AD.py` is `tutorial_CN.py` with
one block added: a fraction of excitatory neurons per area, taken from the amyloid-PET map of
Part 4, is replaced by lower-threshold units. Same anatomy, same drive, same seed.

In [ ]:
ad = (EXPERIMENTS / "tutorial_AD.py").read_text()
start = ad.index("# ---- pathology")
print(ad[start:start + 1500])

In [ ]:
# ~3 minutes again.
print("tutorial_AD  (amyloid-burdened cortex)")
total = 0.0
for stage in ("network", "simulate", "analyse"):
    total += run_stage(stage, "tutorial_AD")
print(f"  {'total':9s} {total:6.1f} s")

h_cn, h_ad = hashes("tutorial_CN"), hashes("tutorial_AD")
print(f"\nnetwork hash  CN {h_cn['network']}\n              AD {h_ad['network']}")
print("-> different parameters, different network, different directory."
      if h_cn["network"] != h_ad["network"] else "-> identical")

## 7.1 Did anything change? Check the working point first

In [ ]:
summary = pd.DataFrame({
    label: {
        "E rate (Hz)": rates(exp).xs("E", level=pop).mean(),
        "I rate (Hz)": rates(exp).xs("I", level=pop).mean(),
        "all (Hz)":    rates(exp).mean(),
    }
    for label, exp in [("CN", "tutorial_CN"), ("AD", "tutorial_AD")]
}).round(3)
summary["change"] = ((summary.AD - summary.CN) / summary.CN).map("{:+.1%}".format)
summary

The excitatory rate rises substantially. This is the model reproducing **network
hyperexcitability** - and nothing was tuned to make it happen. Lowering the threshold of a subset of
excitatory neurons raises their firing, which raises recurrent excitation, which recruits more
neurons; inhibition partly compensates, which is why the inhibitory rate rises too.

Note what did *not* happen: the network did not collapse into synchrony. It is operating at a higher
rate in the *same* asynchronous irregular regime. A pathology that merely made everything
synchronous would be far less interesting, and far less like the data.

## 7.2 Where did it change? The connectome redistributes the pathology

In [ ]:
e_cn = rates("tutorial_CN").xs("E", level=pop).groupby(level="area").mean()
e_ad = rates("tutorial_AD").xs("E", level=pop).groupby(level="area").mean()
burden = per_area.reindex(e_cn.index).fillna(0.0)
delta = e_ad - e_cn
order = delta.sort_values(ascending=False).index

fig, axes = plt.subplots(1, 2, figsize=(15, 5), gridspec_kw={"width_ratios": [2, 1]})
x = np.arange(len(order))
axes[0].bar(x - 0.2, e_cn.loc[order], width=0.4, color="#3b6ea5", label="CN")
axes[0].bar(x + 0.2, e_ad.loc[order], width=0.4, color="#c0392b", label="AD")
axes[0].set_xticks(x); axes[0].set_xticklabels(order, rotation=90, fontsize=7)
axes[0].set_ylabel("mean excitatory rate (Hz)")
axes[0].set_title("Excitatory rate per area"); axes[0].legend()

axes[1].scatter(burden.loc[order], delta.loc[order], s=28, c="#c0392b", alpha=0.85)
axes[1].axhline(0, c="k", lw=0.8)
axes[1].set_xlabel("local amyloid burden")
axes[1].set_ylabel("rate increase AD - CN (Hz)")
axes[1].set_title("local burden vs local change\n"
                  f"Pearson r = {np.corrcoef(burden.loc[order], delta.loc[order])[0, 1]:.2f}")
plt.tight_layout(); plt.show()

**The right-hand panel is the result.** If the model were 34 independent areas, the rate increase
would be a deterministic function of the local amyloid burden and the points would fall on a line.
They do not. Areas with little local pathology are driven up by their connections to heavily
affected hubs; heavily affected areas in weakly connected neighbourhoods rise less than their burden
predicts.

**The connectome redistributes the pathology.** That is the reason to build a network model rather
than a regional one, and the message to carry back to your own field: amyloid-PET tells you where
the protein is; the connectome tells you where the dysfunction goes.

You can see the same thing in the pipeline's own figures - compare the two area overviews, and look
at the amyloid-specific raster, which colours affected neurons separately.

In [ ]:
show("tutorial_AD", "raster/raster_Abeta_precuneus.png", width=680)

## 7.3 The measurement that decides what is real

Everything so far comes from `master_seed = 2903`. The seed sets the connectivity draw, the weight
and delay draws, the Poisson background and the initial membrane potentials - so a different seed is
*a different sample of the same statistical model*. The seed-to-seed spread is the noise floor
against which any claimed effect must be judged.

This is where the hashes earn their keep. Changing only `master_seed` leaves `net_params` untouched,
so the **network hash is unchanged** and stage 1 can be skipped: we copy the existing
`.network_hash` and run only the simulation. Two extra seeds per condition, about 80 seconds.

In [ ]:
import shutil
from eta_ext_search import read_population_rates          # tutorial's own helper

RUN_SEED_CHECK = True
EXTRA_SEEDS = [1234, 5678]


def seed_variant(experiment, seed):
    """An experiment identical to `experiment` but with another seed."""
    name = f"{experiment}_seed{seed}"
    text = (EXPERIMENTS / f"{experiment}.py").read_text()
    assert '"master_seed": 2903' in text
    (EXPERIMENTS / f"{name}.py").write_text(
        text.replace('"master_seed": 2903', f'"master_seed": {seed}'))
    # net_params is untouched, so stage 1's result is valid as it stands.
    shutil.copy(EXPERIMENTS / f"{experiment}.network_hash",
                EXPERIMENTS / f"{name}.network_hash")
    return name


rows = []
if RUN_SEED_CHECK:
    for label, experiment in [("CN", "tutorial_CN"), ("AD", "tutorial_AD")]:
        for seed in [2903] + EXTRA_SEEDS:
            if seed == 2903:
                r = rates(experiment)
            else:
                name = seed_variant(experiment, seed)
                run_stage("simulate", name, quiet=True)
                # The analysis stage was skipped, so read the rates from the
                # spike files directly (1000-5000 ms, as in ana_params).
                r = read_population_rates(sim_dir(name), 1000.0, 5000.0)
            lvl = r.index.names[-1]
            rows.append(dict(condition=label, seed=seed,
                             E_rate=r.xs("E", level=lvl).mean()))
    seeds_df = pd.DataFrame(rows)
    display(seeds_df.round(3))
else:
    print("Seed check skipped.")

In [ ]:
if RUN_SEED_CHECK:
    fig, ax = plt.subplots(figsize=(6, 4.5))
    for x, (cond, color) in enumerate([("CN", "#3b6ea5"), ("AD", "#c0392b")]):
        v = seeds_df.loc[seeds_df.condition == cond, "E_rate"].values
        ax.scatter(np.full_like(v, x, dtype=float), v, s=60, color=color, zorder=3,
                   label=f"{cond} (one point per seed)")
        ax.hlines(v.mean(), x - 0.22, x + 0.22, color=color, lw=2.5, zorder=2)
    ax.set_xticks([0, 1]); ax.set_xticklabels(["CN", "AD"]); ax.set_xlim(-0.5, 1.5)
    ax.set_ylabel("mean excitatory rate (Hz)")
    ax.set_title("Is the effect bigger than the noise?")
    ax.legend(fontsize=8); ax.margins(y=0.35)
    plt.tight_layout(); plt.show()

    cn = seeds_df.loc[seeds_df.condition == "CN", "E_rate"].values
    ad = seeds_df.loc[seeds_df.condition == "AD", "E_rate"].values
    spread = max(cn.std(ddof=1), ad.std(ddof=1))
    effect = ad.mean() - cn.mean()
    print(f"CN {cn.mean():.3f} +/- {cn.std(ddof=1):.3f} Hz")
    print(f"AD {ad.mean():.3f} +/- {ad.std(ddof=1):.3f} Hz")
    print(f"difference {effect:+.3f} Hz = {abs(effect) / spread:.1f} x the seed spread")
    print("\n(3 seeds is the smallest number that gives a spread at all; "
          "a real analysis uses 10-20.)")

### What you may and may not say

The rate effect clears the seed spread by a wide margin, so it is a property of the model rather
than of this particular random draw.

Apply the same test before believing anything else. Run the same comparison on the spectra, for
instance, and you will find that the shift in band power between CN and AD at this network size is
*comparable to the seed-to-seed variability* - which means this configuration cannot support a claim
about spectral slowing, however much the literature would like it to. That is a finding about your
measurement, not a failure of the model, and reporting it is the difference between a result and a
figure.

- **Supported here:** amyloid-affected neurons drive the network to a higher firing rate, and the
  connectome redistributes that increase well beyond the areas carrying the pathology.
- **Not supported here:** anything requiring a few-percent effect - spectral slowing, functional
  connectivity changes. Those need the full-scale network, tens of seconds of simulated time, and
  enough seeds for an error bar - the multi-seed sweep of Part 9, run as a cluster job.

Knowing which of your figures survives a seed change is the single most useful habit this tutorial
can give you.

## 7.4 One step further: an individual, from their own scans

*Optional, about 3.5 minutes. Set `RUN_SUBJECT = True` below.*

Everything so far mixes sources: a group-average connectome, with one participant's amyloid map
applied to that generic brain. The model can do better, because all three of its inputs can come
from the same person:

| Input | This person's data | Parameter |
|---|---|---|
| **Structural connectivity** | diffusion-MRI tractography, 34x34 per hemisphere | `net_params["predictive_connectomic_params"]["con_path"]` |
| **Amyloid distribution** | amyloid-PET, per DK area | `net_params["Abeta_ratio_E"]` |
| **Functional connectivity** | resting-state BOLD, 835 frames | `ana_params["experimental_fc"]` |

`model/experimental_data/ADNI/` bundles exactly that for ADNI participant `023_S_6661`, and
`experiments/tutorial_subject.py` wires the three together. The structural file has the same variable
layout as the group-average one, so swapping it is a one-line change - and because it changes
`net_params`, the network hash changes and stage 1 rebuilds on this person's wiring.

The payoff is the comparison: simulated functional connectivity becomes a prediction about *this
person*, checkable against *their own* measured fMRI rather than a group average.

In [ ]:
RUN_SUBJECT = False        # <-- set True to run it (network + simulate + analyse, ~3.5 min)

if RUN_SUBJECT:
    print("tutorial_subject  (ADNI participant 023_S_6661)")
    for stage in ("network", "simulate", "analyse"):
        run_stage(stage, "tutorial_subject")

    r = rates("tutorial_subject")
    lvl = r.index.names[-1]
    print(f"\n  E {r.xs('E', level=lvl).mean():.2f} Hz   "
          f"I {r.xs('I', level=lvl).mean():.2f} Hz")
    show("tutorial_subject", "simulated_fc.png", width=520)
    show("tutorial_subject", "experimental_fc.png", width=520)
else:
    print("Skipped. Set RUN_SUBJECT = True to run the individualised experiment.")

### What this run shows, and the trap in it

Measured on this participant, next to the two runs you already have:

| Connectome | Amyloid | Mean excitatory rate |
|---|---|---|
| group average (HCP) | none | 3.02 Hz |
| group average (HCP) | population map, mean burden 0.154 | 4.70 Hz |
| **this person** | none | **3.34 Hz** |
| **this person** | **their own map, mean burden 0.267** | **7.91 Hz** |

Two things follow. First, **the individual connectome barely moves the working point** (3.02 to
3.34 Hz) while **their heavier amyloid burden moves it a great deal** (3.34 to 7.91 Hz). Individual
anatomy matters less here than individual pathology - worth knowing before you invest in per-subject
tractography.

Second, and more important: **7.91 Hz is outside the 3-5 Hz band the drive was calibrated for.** By
the rule in Part 3.4 you should not read that run's spectra or functional connectivity at face
value; the network is being pushed well past the operating point the rest of the tutorial assumes.
The rate itself is a legitimate read-out. Everything downstream of it is suspect.

The fix is the one Part 3 already taught - recalibrate for this configuration:

```bash
python tools/eta_ext_search.py \
    --con-path    model/experimental_data/ADNI/023_S_6661_sc.mat \
    --abeta-ratio model/experimental_data/ADNI/023_S_6661_abeta_ratio.csv
```

then put the `eta_ext` it reports into `experiments/tutorial_subject.py` and run again. Whether a
patient-specific model *should* be recalibrated per patient - or whether the drive should be held
fixed so that the rate difference **is** the result - is a real modelling decision, not a settled
one. Making that choice, and defending it, is exercise 9.

---


# Part 8 - Beyond the Laptop

The downscaled model is a development instrument. Quantitative results - functional connectivity
above all, because it is a collective property - need the full-scale network.

## 8.1 The same three stages, wired by Snakemake

On a cluster you do not call the three commands by hand. A Snakefile declares them as rules whose
inputs and outputs are the hash files, and Snakemake works out what needs rebuilding:

```python
rule createNetwork:
    input:  "experiments/{experiment}.py"
    output: "experiments/{experiment}.network_hash"
    shell:  "python src/snakemake_network.py {input} {output} {log}"

rule simulateNetwork:
    input:  "experiments/{experiment}.py", "experiments/{experiment}.network_hash"
    output: "experiments/{experiment}.simulation_hash"
    threads: 16
    # srun starts one MPI rank per SLURM task; NEST uses {threads} OpenMP
    # threads inside each rank
    shell:  "srun python src/snakemake_simulation.py {input} {output} {threads} {log}"

rule analyzeNetwork:
    input:  "experiments/{experiment}.py", "experiments/{experiment}.network_hash",
            "experiments/{experiment}.simulation_hash"
    output: "experiments/{experiment}.analysis_hash"
    shell:  "python src/snakemake_analysis.py {input} {output} {BASE_PATH} {log}"
```

That is the entire difference between what you ran above and a production run: the same three
commands, plus `srun` for MPI and a dependency graph. Edit an experiment file, type `snakemake`, and
only the affected stages re-run - this is what the hashes are for.

## 8.2 Sizing the job

```
synapses(N_scaling, K_scaling) = 4.278e10 x N_scaling x K_scaling
peak RAM                       = 0.4 GB   + 128 bytes x synapses
```

| Configuration | Synapses | Aggregate RAM | Typical allocation |
|---|---|---|---|
| `N=0.005, K=0.1` (this tutorial) | 2.1e7 | 3 GB | a laptop |
| `N=0.1, K=0.1` (the repository's `local_test`) | 4.3e8 | ~51 GB | one fat node |
| `N=0.5, K=0.5` | 1.1e10 | ~1.3 TB | ~11 nodes @ 128 GB |
| `N=1.0, K=1.0` (full scale) | 4.3e10 | **~5 TB** | ~40 nodes @ 128 GB |

NEST distributes connections across MPI ranks, so what matters is the **aggregate** RAM, plus 20-30%
headroom for the spike-communication buffers allocated during `Prepare`.

Three pieces of advice, all learned the expensive way:

1. **Never jump straight to full scale.** Run `N=0.1, K=0.1` on one node first. It exercises every
   code path, finishes in minutes, and catches every path, module and environment error.
2. **Split simulation and analysis into separate jobs.** Simulation is MPI-parallel and
   memory-hungry; analysis is a single process that reads spike files and is I/O-bound. Sizing them
   together wastes a large allocation on a serial tail.
3. **Watch the spike output.** Full scale produces hundreds of GB of ASCII spike files. Write to
   scratch, not to your home quota, and delete failed runs.

## 8.3 The batch job

`cluster/submit_model.sh` is a self-contained SLURM job. Submitted as it is, it:

```
sbatch submit_model.sh
   |
   +-- 1. creates a fresh run directory under $WORKROOT
   +-- 2. git clone's the model repository - a clean checkout at a named commit
   +-- 3. writes full-scale CN and AD experiment files and a Snakefile into it
   +-- 4. activates the environment and sources NEST's nest_vars.sh
   +-- 5. runs snakemake: createNetwork -> simulateNetwork (srun) -> analyzeNetwork
   +-- 6. prints the hashes so you can find the output
```

Cloning inside the job is deliberate, and worth adopting as a habit: the run is reproducible from a
named commit rather than from whatever happened to be in your home directory, concurrent jobs cannot
interfere through a shared working tree, and the job log contains everything needed to reproduce it.
It also has to write the experiment files itself, because the repository's `.gitignore` excludes
`experiments/` and `Snakefile` - they are per-run scratch by design.

In [ ]:
script = (TUTORIAL / "cluster" / "submit_model.sh").read_text()
head = script.split("# ----------------------------------------------------------------- settings --")
print(head[0][-1200:])
print("# --- settings ---" + head[1][:1500])
print(f"\n... {len(script.splitlines())} lines in total: "
      f"{(TUTORIAL / 'cluster' / 'submit_model.sh')}")

## 8.4 Submitting and monitoring

```bash
scp cluster/submit_model.sh user@cluster:~/
ssh user@cluster
mkdir -p logs
sbatch submit_model.sh

# override any setting without editing the file
sbatch --export=ALL,N_SCALING=0.1,K_SCALING=0.1,T_SIM=5000.0,ETA_EXT=3.45 submit_model.sh

squeue -u $USER
tail -f logs/cortex-model-<jobid>.out
sacct -j <jobid> --format=JobID,JobName,Elapsed,MaxRSS,State
```

`sacct`'s **`MaxRSS`** is the number to write down: your measured memory per rank, and how you size
the next, larger run.

| Scheduler | Directives | Submit | Launcher |
|---|---|---|---|
| SLURM | `#SBATCH` | `sbatch` | `srun` |
| PBS / Torque | `#PBS` | `qsub` | `mpirun -np $PBS_NP` |
| Fujitsu PJM (Fugaku) | `#PJM` | `pjsub` | `mpiexec` |
| LSF | `#BSUB` | `bsub <` | `mpirun` |

Only the directive block and the launcher change; the clone-generate-run body is portable.

**First-run checklist**

- [ ] `python -c "import nest"` works **on a compute node**, not only on the login node
- [ ] the clone URL is reachable from compute nodes - many clusters have no outbound network there,
      in which case clone on the login node and point `MODEL_REPO` at that path
- [ ] `$WORKROOT` is on scratch with room for hundreds of GB
- [ ] an `N=0.1, K=0.1` run completed before you request a large allocation
- [ ] ranks x threads matches `--nodes` x `--ntasks-per-node` x `--cpus-per-task`

> **A login-node trap that costs people hours.** On some clusters `import numpy` hangs forever on a
> login node because of thread-affinity defaults. Set `OPENBLAS_NUM_THREADS=4` before any Python
> invocation - the batch script already exports it.

---

# Part 9 - Exercises

Every one of these is an edit to an experiment file plus a pipeline run. That is the point: you now
have the whole interface.

### A. Warm-up (minutes each)

1. **Dose-response.** Copy `tutorial_AD.py` and replace the CSV with a scalar
   `net_params["Abeta_ratio_E"] = 0.05` (then 0.1, 0.2, 0.3). Plot mean excitatory rate against
   burden. Is it linear? Is there a burden beyond which the network destabilises?
2. **Spatial specificity.** Same *total* burden, different geometry: all of it in one area
   (`{"precuneus": 0.9, "default": 0.0}`) versus spread uniformly (`0.154`). Which produces the
   larger global change? What does that say about the clinical relevance of amyloid *topography*
   versus amyloid *load*?
3. **Inhibitory pathology.** Set `Abeta_ratio_I` to a non-zero value. Damaging inhibition should push
   the network the same way as damaging excitation - but check the magnitude, and at what point it
   runs away.

### B. Intermediate

4. **Calibration.** Run `tools/eta_ext_search.py` at your own `N_scaling`/`K_scaling` and read its
   figure. How sharp is the transition from silence to activity? Why is the rate curve almost flat
   between `eta_ext` 1.9 and 2.8 before climbing again - what is holding it down?
5. **Gamma entrainment.** Uncomment the `stimuli` block in `tutorial_CN.py` to drive 40 Hz into the
   precuneus (the GENUS paradigm; Iaccarino et al. 2016). Compare `lfp_proxy_psd_layer_II_III.png`
   with and without it: how much 40 Hz power appears in areas you did *not* stimulate? Then repeat on
   `tutorial_AD.py` - does amyloid make the network easier or harder to entrain?
6. **Frequency sweep.** Repeat exercise 5 at 10, 20, 40 and 80 Hz. Does the network have a preferred
   frequency, and does it match the spontaneous gamma peak?
7. **Longer runs.** Raise `t_sim` to 20000 ms and re-run the analysis only. How much does the
   simulated-vs-experimental FC correlation improve? This is the cheapest way to see why 5 s was not
   enough.

### C. Project-scale

8. **Fix the downscaling.** Part 3.2 showed the DC compensation is evaluated from already-scaled
   quantities. Change `extraDCforScaledNetwork()` to use the unscaled ones and check whether the
   downscaled network then reproduces full-scale rates at `eta_ext = 1.1`. Compare against the
   mean-field prediction (`tools/eta_ext_search.py --method theory`) rather than only against
   simulation.
9. **Patient-specific modelling.** Run `experiments/tutorial_subject.py` (Part 7.4), then decide what
   to do about its working point. Recalibrate `eta_ext` for this participant with
   `tools/eta_ext_search.py --con-path ... --abeta-ratio ...` and re-run: does the
   simulated-versus-own-FC correlation improve once the network is back in its intended regime? Then
   argue the other side - if you recalibrate per patient, you have removed the hyperexcitability you
   set out to measure. Which comparison answers which question?
10. **Multi-seed statistics.** Extend the Part 7.3 loop to 10-20 seeds per condition and put proper
    error bars on every number in Parts 6 and 7. Only then is any of it a result - and at that point
    it belongs in a batch job rather than a notebook.

---

# Part 10 - Troubleshooting

| Symptom | Likely cause | Fix |
|---|---|---|
| `The bundled model is missing` | the notebook was moved away from its folder | run it from the tutorial folder, so `model/` sits beside it |
| `ModuleNotFoundError: No module named 'nest'` | wrong kernel, or NEST not installed | check the kernel selector; `python -c "import nest"` in the same environment. On Windows you must be inside WSL2 |
| A stage raises `FileNotFoundError` for `experimental_data/...` | the subprocess ran in the wrong directory | `run_stage` passes `cwd=MODEL_ROOT`; do not change that - the repository resolves data paths from the working directory |
| Mean firing rate is ~0 | working point not calibrated | Part 3. Run `tools/eta_ext_search.py` |
| Rates look plausible but nothing changes when you edit a parameter | you edited a different file, or the hash did not change | check `hashes(...)` before and after; a changed `net_params` must change the network hash |
| Editing `ana_params` changes nothing | cached `.pkl` in the run directory | delete `rates.pkl` / `spikes.pkl` / `lfp_proxy_kernel.pkl`, or change a parameter that alters the analysis hash |
| Kernel dies during the simulate stage | out of RAM | lower `N_scaling` in the experiment file (Part 2) |
| A cell runs for many minutes with no output | swapping to disk | same as above |
| `TypeError: Type of specific_dict[x] ... dont match` | an override has a different type from the default it replaces | usually an `int` where the default is a `float`, or vice versa |
| `ValueError: unknown area ...` | area name not in the DK list | lower-case, no spaces, e.g. `superiorfrontal` |
| Figures look empty | matplotlib style file not found | the analysis stage resolves it relative to the repository root; it must run with `cwd=MODEL_ROOT` |

### Starting over

```python
import shutil
shutil.rmtree(RESULTS, ignore_errors=True)               # all results
for f in EXPERIMENTS.glob("*_hash"):                     # and the stage markers
    f.unlink()
```

The model repository is never touched by any of this.

---

# Part 11 - Where to Go Next

### In this folder

| Path | What it is |
|---|---|
| `experiments/` | one file per experiment - the interface you have been using |
| `tools/eta_ext_search.py` | the drive calibration of Part 3 |
| `model/src/default_*_params.py` | every parameter and its default, with comments. The reference manual |
| `model/src/network.py` | network construction, weights, downscaling, DC compensation |
| `model/src/simulation.py` | NEST setup, population creation, connection routines |
| `model/src/analysis.py` | the analysis suite - the best documentation of what can be measured |
| `model/src/theory/rates.py` | NNMT mean-field solver: rates without simulating |
| `model/experimental_data/` | every dataset the model reads, with a README per source |

### Key references

**The model**

- Pronold, J., van Meegen, A., Shimoura, R. O., Vollenbröker, H., Senden, M., Hilgetag, C. C.,
  Bakker, R., & van Albada, S. J. (2024). Multi-scale spiking network model of human cerebral
  cortex. *Cerebral Cortex*. https://doi.org/10.1093/cercor/bhae409
- Schmidt, M., Bakker, R., Shen, K., Bezgin, G., Diesmann, M., & van Albada, S. J. (2018). A
  multi-scale layer-resolved spiking network model of resting-state dynamics in macaque visual
  cortical areas. *PLoS Computational Biology*, 14(10), e1006359.

**Downscaling**

- van Albada, S. J., Helias, M., & Diesmann, M. (2015). Scalability of asynchronous networks is
  limited by one-to-one mapping between effective connectivity and correlations. *PLoS Computational
  Biology*, 11(9), e1004490.
- Potjans, T. C., & Diesmann, M. (2014). The cell-type specific cortical microcircuit. *Cerebral
  Cortex*, 24(3), 785-806.

**Alzheimer's disease dynamics**

- Palop, J. J., & Mucke, L. (2016). Network abnormalities and interneuron dysfunction in Alzheimer
  disease. *Nature Reviews Neuroscience*, 17(12), 777-792.
- Jeong, J. (2004). EEG dynamics in patients with Alzheimer's disease. *Clinical Neurophysiology*,
  115(7), 1490-1505.
- Iaccarino, H. F., et al. (2016). Gamma frequency entrainment attenuates amyloid load and modifies
  microglia. *Nature*, 540, 230-235.

**Haemodynamics and whole-brain modelling**

- Buxton, R. B., Wong, E. C., & Frank, L. R. (1998). Dynamics of blood flow and oxygenation changes
  during brain activation: the balloon model. *Magnetic Resonance in Medicine*, 39(6), 855-864.
- Deco, G., Jirsa, V. K., & McIntosh, A. R. (2011). Emerging concepts for the dynamical organization
  of resting-state activity in the brain. *Nature Reviews Neuroscience*, 12(1), 43-56.

**Tools**

- NEST: https://nest-simulator.readthedocs.io
- NNMT (mean-field toolbox): https://nnmt.readthedocs.io
- Snakemake: https://snakemake.readthedocs.io

---

## Summary

| | |
|---|---|
| What the model is | 34 areas x 4 layers x E/I, anatomy from cytoarchitecture and tractography, no free parameters fitted to the output |
| How you use it | one experiment file, three stages, hash-addressed results |
| What you ran | a healthy and an amyloid-burdened cortex, 5 s each, ~200 figures per run |
| What it showed | network hyperexcitability, redistributed by the connectome, surviving a seed check |
| What it could not show | few-percent effects - spectra, functional connectivity - at this size and duration |
| How to go further | longer runs, full scale, many seeds: `cluster/submit_model.sh` |

Two habits are worth more than any figure in this notebook. **A downscaled network is a different
model** - verify its working point before interpreting anything. **An effect that does not survive a
change of random seed is not an effect** - measure your noise floor before naming a finding.